# 05 — LSTM Autoencoder

## 1. Objective

Develop an LSTM autoencoder for unsupervised anomaly detection on the SWaT multivariate time series.

The model will be trained using normal calibration data and will learn to reconstruct temporal patterns in the sensor measurements. Reconstruction error will then provide an anomaly score: observations that are poorly reconstructed relative to the learned normal behaviour may indicate anomalous behaviour.

The implementation follows the LSTM autoencoder approach used in the primary reference notebook, while maintaining the project's requirements for chronological data handling, separation of calibration and evaluation data, reproducibility, and prevention of data leakage.

Attack labels will not be used during model training or reconstruction-error threshold calibration.


## 2. Methodology

An autoencoder is trained to reconstruct input observations from a compressed internal representation. When trained only on normal process behaviour, the model is expected to learn the temporal patterns present in normal operation.

For an input sequence X, the LSTM encoder maps the sequence to a lower-dimensional representation:

X → Encoder → Z

The decoder then reconstructs the sequence:

Z → Decoder → X̂

The reconstruction error measures the difference between the original and reconstructed sequence. For a sequence containing T time steps and D sensor features, the mean squared reconstruction error is:

MSE = (1 / (T × D)) × Σ (Xₜ,ⱼ − X̂ₜ,ⱼ)²

where:

- Xₜ,ⱼ is the observed value for feature j at time step t.
- X̂ₜ,ⱼ is the corresponding reconstructed value.
- T is the sequence length.
- D is the number of sensor features.

The reconstruction error is used as the anomaly score. A threshold established from normal calibration behaviour can then be used to classify sequences as normal or anomalous.

### Temporal Windowing

The model operates on sequences rather than isolated observations. Each sequence contains a fixed number of consecutive observations while preserving their chronological order.

The primary reference implementation uses a window size of 30 observations. This project will initially retain that value so that the implementation remains comparable to the reference, unless the data or modelling results provide evidence that a change is necessary.

Training sequences will be generated from normal calibration data only. Evaluation sequences will remain temporally separate from the training data.

### Data Leakage Prevention

The preprocessing parameters used by the model must be learned from the normal calibration data only. Validation and attack observations must not influence the fitted scaler or the model parameters during training.

The final anomaly threshold will also be established without using attack labels for threshold selection.

## 3. Reproducibility and Model Configuration

The LSTM autoencoder will use fixed random seeds and explicit model parameters so that the experiment can be reproduced.

The initial configuration follows the primary reference implementation:

| Parameter | Value |
|:--|--:|
| Random seed | 42 |
| Sequence window | 30 observations |
| Training stride | 5 |
| Evaluation stride | 1 |
| Hidden size | 64 |
| LSTM layers | 2 |
| Bottleneck size | 16 |
| Dropout | 0.2 |
| Loss function | Mean Squared Error |
| Optimizer | Adam |
| Initial learning rate | 0.001 |
| Batch size | 256 |
| Learning-rate reduction factor | 0.5 |
| Learning-rate scheduler patience | 3 epochs |
| Early-stopping patience | 5 epochs |

These values provide the initial model configuration and are not being treated as universally optimal hyperparameters.

Any change to the configuration will be recorded and justified by the observed modelling results rather than by tuning against final attack labels.